# Basic RAG Experiment - LiveVectorLake

This notebook demonstrates a complete end-to-end RAG (Retrieval-Augmented Generation) pipeline using local documents, vector databases, and Ollama LLMs.

## Experiment Overview
- **Goal**: Build and test a basic RAG system with chunking, embedding, vector storage, retrieval, and generation
- **Vector DB**: Qdrant (primary)
- **Embeddings**: SentenceTransformers
- **LLM**: Ollama (Llama3/Mistral/Gemma)
- **Data**: Local documents from `data/` folder

## Pipeline Steps
1. Environment setup and package installation
2. Document loading and preview
3. Text chunking with metadata
4. Embedding generation
5. Vector database setup and indexing
6. Query processing and retrieval
7. RAG answer generation
8. Results analysis and future improvements

## Step 1: Environment Setup and Package Installation

First, we'll check and install all required packages for our RAG pipeline.

In [16]:
# Check and install required packages
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Required packages for RAG pipeline
packages = [
    "sentence-transformers",
    "qdrant-client",
    "ollama",
    "numpy",
    "pandas",
    "requests"
]

for package in packages:
    try:
        __import__(package.replace("-", "_"))
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        install_package(package)

✓ sentence-transformers already installed
✓ qdrant-client already installed
✓ ollama already installed
✓ numpy already installed
✓ pandas already installed
✓ requests already installed


In [17]:
import sys
!{sys.executable} -m pip install sentence-transformers qdrant-client ollama


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.1.2 -> 25.2
[notice] To update, run: C:\Users\asus\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [18]:
# Import all necessary libraries
import os
import json
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import ollama

print("All imports successful!")

All imports successful!


## Step 2: Document Loading and Preview

Load sample documents from the `data/` folder. Since the example.txt is empty, we'll create some sample content for demonstration.

In [19]:
# Create sample documents for demonstration
data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)

# Sample documents about AI and machine learning
sample_docs = {
    "ai_basics.txt": """
Artificial Intelligence (AI) is a branch of computer science that aims to create intelligent machines.
Machine learning is a subset of AI that enables computers to learn and improve from experience without being explicitly programmed.
Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.
Natural language processing (NLP) helps computers understand, interpret, and generate human language.
""",
    "vector_databases.txt": """
Vector databases are specialized databases designed to store and query high-dimensional vectors efficiently.
They use similarity search algorithms like cosine similarity and Euclidean distance to find related vectors.
Popular vector databases include ChromaDB, Qdrant, Pinecone, and Weaviate.
Vector databases are essential for applications like semantic search, recommendation systems, and RAG pipelines.
""",
    "rag_systems.txt": """
Retrieval-Augmented Generation (RAG) combines information retrieval with text generation.
RAG systems first retrieve relevant documents from a knowledge base, then use them to generate informed responses.
The retrieval component typically uses vector similarity search to find relevant context.
RAG helps reduce hallucinations in large language models by grounding responses in factual information.
"""
}

# Write sample documents
for filename, content in sample_docs.items():
    with open(data_dir / filename, 'w') as f:
        f.write(content.strip())

print(f"Created {len(sample_docs)} sample documents in {data_dir}")

Created 3 sample documents in ..\data


In [20]:
# Load and preview documents
documents = {}

for file_path in data_dir.glob("*.txt"):
    if file_path.stat().st_size > 0:  # Skip empty files
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
            documents[file_path.name] = content
            print(f"\n📄 {file_path.name} ({len(content)} chars)")
            print(f"Preview: {content[:100]}...")

print(f"\nLoaded {len(documents)} documents")


📄 ai_basics.txt (442 chars)
Preview: Artificial Intelligence (AI) is a branch of computer science that aims to create intelligent machine...

📄 example.txt (38 chars)
Preview: This file is intentionally left blank....

📄 rag_systems.txt (398 chars)
Preview: Retrieval-Augmented Generation (RAG) combines information retrieval with text generation.
RAG system...

📄 vector_databases.txt (405 chars)
Preview: Vector databases are specialized databases designed to store and query high-dimensional vectors effi...

Loaded 4 documents


## Step 3: Text Chunking with Metadata

Split documents into smaller chunks for better retrieval granularity. We'll use sentence-based chunking with overlap.

In [21]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 50) -> List[str]:
    """Simple sentence-based chunking with overlap"""
    sentences = text.split('. ')
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        if len(current_chunk + sentence) < chunk_size:
            current_chunk += sentence + ". "
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence + ". "
    
    if current_chunk:
        chunks.append(current_chunk.strip())
    
    return chunks

# Create chunks with metadata
all_chunks = []
chunk_metadata = []

for filename, content in documents.items():
    chunks = chunk_text(content)
    
    for i, chunk in enumerate(chunks):
        chunk_id = f"{filename}_{i}"
        all_chunks.append(chunk)
        chunk_metadata.append({
            "chunk_id": chunk_id,
            "source_file": filename,
            "chunk_index": i,
            "content_date": datetime.now().isoformat(),
            "char_count": len(chunk)
        })

print(f"Created {len(all_chunks)} chunks from {len(documents)} documents")
print(f"\nSample chunk:")
print(f"ID: {chunk_metadata[0]['chunk_id']}")
print(f"Content: {all_chunks[0]}")
print(f"Metadata: {chunk_metadata[0]}")

Created 4 chunks from 4 documents

Sample chunk:
ID: ai_basics.txt_0
Content: Artificial Intelligence (AI) is a branch of computer science that aims to create intelligent machines.
Machine learning is a subset of AI that enables computers to learn and improve from experience without being explicitly programmed.
Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.
Natural language processing (NLP) helps computers understand, interpret, and generate human language..
Metadata: {'chunk_id': 'ai_basics.txt_0', 'source_file': 'ai_basics.txt', 'chunk_index': 0, 'content_date': '2025-09-17T22:58:45.171973', 'char_count': 443}


## Step 4: Embedding Generation

Convert text chunks into vector embeddings using SentenceTransformers. We'll use a lightweight model suitable for semantic similarity.

In [22]:
# Initialize embedding model
print("Loading SentenceTransformer model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  # Lightweight, fast model
print(f"Model loaded. Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

# Generate embeddings for all chunks
print(f"\nGenerating embeddings for {len(all_chunks)} chunks...")
embeddings = embedding_model.encode(all_chunks, show_progress_bar=True)

print(f"Generated embeddings shape: {embeddings.shape}")
print(f"Sample embedding (first 5 dimensions): {embeddings[0][:5]}")

Loading SentenceTransformer model...
Model loaded. Embedding dimension: 384

Generating embeddings for 4 chunks...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.62it/s]

Generated embeddings shape: (4, 384)
Sample embedding (first 5 dimensions): [-0.04844885 -0.04057992  0.03953846  0.02492553  0.01154871]


## Step 5: Vector Database Setup and Indexing

Set up Qdrant as our vector database and index all chunks with their embeddings and metadata.

In [23]:
# Initialize Qdrant client
qdrant_client = QdrantClient(":memory:")  # In-memory for demo, use path for persistence

# Create collection
collection_name = "rag_experiment"
try:
    qdrant_client.delete_collection(collection_name)  # Reset for fresh start
except:
    pass

qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=embeddings.shape[1], distance=Distance.COSINE)
)

print(f"Created Qdrant collection: {collection_name}")

Created Qdrant collection: rag_experiment


In [24]:
# Add documents to collection
points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={**chunk_metadata[i], "document": all_chunks[i]}
    )
    for i in range(len(all_chunks))
]

qdrant_client.upsert(
    collection_name=collection_name,
    points=points
)

print(f"Added {len(all_chunks)} documents to Qdrant collection")
info = qdrant_client.get_collection(collection_name)
print(f"Collection vectors count: {info.vectors_count}")

# Verify vectors were added
try:
    # Test search to confirm vectors exist
    test_embedding = embedding_model.encode(["test query"])
    search_results = qdrant_client.search(
        collection_name=collection_name,
        query_vector=test_embedding.tolist(),
        limit=1
    )
    print(f"✓ Vectors successfully added - found {len(search_results)} results in test search")
    if search_results:
        print(f"Sample result: {search_results[0].payload['chunk_id']}")
except Exception as e:
    print(f"❌ Error verifying vectors: {e}")


Added 4 documents to Qdrant collection
Collection vectors count: None
❌ Error verifying vectors: Multivector  is not found in the collection


C:\Users\asus\AppData\Local\Temp\ipykernel_36596\1838821408.py:24: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_results = qdrant_client.search(


## Step 6: Query Processing and Retrieval

Implement query processing: embed the user query and retrieve the most similar chunks from the vector database.

In [25]:
def retrieve_similar_chunks(query: str, top_k: int = 3) -> Dict[str, Any]:
    """Retrieve top-k similar chunks for a given query"""
    
    # Embed the query
    query_embedding = embedding_model.encode([query])
    
    # Search in Qdrant
    results = qdrant_client.search(
        collection_name=collection_name,
        query_vector=query_embedding.tolist(),
        limit=top_k
    )
    
    return {
        "query": query,
        "retrieved_chunks": [hit.payload["document"] for hit in results],
        "chunk_ids": [hit.payload["chunk_id"] for hit in results],
        "distances": [1 - hit.score for hit in results],  # Convert similarity to distance
        "metadatas": [hit.payload for hit in results]
    }

# Test retrieval with sample queries
test_queries = [
    "What is machine learning?",
    "How do vector databases work?",
    "Explain RAG systems"
]

for query in test_queries:
    print(f"\n🔍 Query: {query}")
    results = retrieve_similar_chunks(query, top_k=2)
    
    for i, (chunk, distance, chunk_id) in enumerate(zip(
        results["retrieved_chunks"], 
        results["distances"], 
        results["chunk_ids"]
    )):
        print(f"  {i+1}. [{chunk_id}] (distance: {distance:.3f})")
        print(f"     {chunk[:100]}...")

C:\Users\asus\AppData\Local\Temp\ipykernel_36596\2606936238.py:8: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant_client.search(



🔍 Query: What is machine learning?


ValueError: Multivector  is not found in the collection

## Step 7: RAG Answer Generation

Use Ollama to generate answers based on retrieved context. We'll create a prompt template that includes the retrieved chunks.

In [ ]:
# Check available Ollama models
try:
    models = ollama.list()
    available_models = [model['name'] for model in models['models']]
    print(f"Available Ollama models: {available_models}")
    
    # Select first available model or default
    if available_models:
        selected_model = available_models[0]
    else:
        selected_model = "llama3"  # Default fallback
        print(f"No models found, will try to use: {selected_model}")
        
except Exception as e:
    print(f"Error connecting to Ollama: {e}")
    print("Make sure Ollama is running locally")
    selected_model = "llama3"

Error connecting to Ollama: 'name'
Make sure Ollama is running locally


In [ ]:
def generate_rag_answer(query: str, top_k: int = 3) -> Dict[str, Any]:
    """Generate RAG answer using retrieved context"""
    
    # Retrieve relevant chunks
    retrieval_results = retrieve_similar_chunks(query, top_k)
    
    # Prepare context from retrieved chunks
    context_chunks = retrieval_results["retrieved_chunks"]
    context = "\n\n".join([f"Context {i+1}: {chunk}" for i, chunk in enumerate(context_chunks)])
    
    # Create RAG prompt
    prompt = f"""
Based on the following context information, please answer the question accurately and concisely.

Context:
{context}

Question: {query}

Answer: Please provide a clear answer based on the context above. If the context doesn't contain enough information, mention that.
"""
    
    try:
        # Generate response using Ollama
        response = ollama.generate(
            model=selected_model,
            prompt=prompt,
            options={
                "temperature": 0.1,  # Low temperature for factual responses
                "top_p": 0.9
            }
        )
        
        return {
            "query": query,
            "context_chunks": context_chunks,
            "chunk_sources": [meta["source_file"] for meta in retrieval_results["metadatas"]],
            "generated_answer": response["response"],
            "model_used": selected_model,
            "retrieval_distances": retrieval_results["distances"]
        }
        
    except Exception as e:
        return {
            "query": query,
            "context_chunks": context_chunks,
            "error": f"Failed to generate answer: {str(e)}",
            "fallback_answer": "Unable to generate answer due to LLM connection issues."
        }

print(f"RAG generation function ready. Using model: {selected_model}")

RAG generation function ready. Using model: llama3


## Step 8: Complete RAG Pipeline Demo

Run the complete RAG pipeline with sample queries and analyze the results.

In [ ]:
# Demo queries
demo_queries = [
    "What is the difference between AI and machine learning?",
    "How do vector databases help with similarity search?",
    "What are the benefits of RAG systems?"
]

for i, query in enumerate(demo_queries, 1):
    print(f"\n{'='*60}")
    print(f"RAG DEMO {i}: {query}")
    print(f"{'='*60}")
    
    # Generate RAG answer
    result = generate_rag_answer(query, top_k=2)
    
    # Display results
    print(f"\n📝 QUERY: {result['query']}")
    
    print(f"\n📚 RETRIEVED CONTEXT:")
    for j, (chunk, source, distance) in enumerate(zip(
        result['context_chunks'], 
        result['chunk_sources'],
        result.get('retrieval_distances', [])
    ), 1):
        print(f"  {j}. Source: {source} (similarity: {1-distance:.3f})")
        print(f"     {chunk}")
        print()
    
    print(f"🤖 GENERATED ANSWER:")
    if 'error' in result:
        print(f"❌ {result['error']}")
        print(f"Fallback: {result.get('fallback_answer', 'No fallback available')}")
    else:
        print(result['generated_answer'])
        print(f"\n(Generated using: {result.get('model_used', 'Unknown model')})")

## Step 9: Results Analysis and Future Improvements

Analyze the RAG pipeline performance and identify areas for improvement.

In [ ]:
# Pipeline statistics
print("📊 RAG PIPELINE STATISTICS")
print(f"{'='*40}")
print(f"Documents processed: {len(documents)}")
print(f"Total chunks created: {len(all_chunks)}")
print(f"Average chunk length: {np.mean([len(chunk) for chunk in all_chunks]):.1f} chars")
print(f"Embedding dimension: {embeddings.shape[1]}")
print(f"Vector database: Qdrant")
print(f"Embedding model: all-MiniLM-L6-v2")
print(f"LLM model: {selected_model}")

# Chunk distribution by source
chunk_distribution = {}
for meta in chunk_metadata:
    source = meta['source_file']
    chunk_distribution[source] = chunk_distribution.get(source, 0) + 1

print(f"\n📄 CHUNK DISTRIBUTION:")
for source, count in chunk_distribution.items():
    print(f"  {source}: {count} chunks")

## Future Improvements and TODOs

Based on this initial RAG experiment, here are key areas for enhancement:

### 🔄 Change Data Capture (CDC)
- **TODO**: Implement document versioning and change tracking
- **TODO**: Add incremental updates to vector database using Qdrant's upsert
- **TODO**: Track document modification timestamps
- **TODO**: Handle document deletions and updates

### 🕒 Temporal Logic
- **TODO**: Add dual-date reasoning (valid-time vs transaction-time)
- **TODO**: Implement time-sensitive query processing
- **TODO**: Support historical document versions
- **TODO**: Add temporal metadata to chunks

### 🔍 Hybrid Retrieval
- **TODO**: Combine vector similarity with keyword search (BM25)
- **TODO**: Implement re-ranking algorithms
- **TODO**: Add query expansion techniques
- **TODO**: Support multi-modal retrieval (text + metadata)

### 📊 Evaluation and Monitoring
- **TODO**: Add retrieval quality metrics (precision@k, recall@k)
- **TODO**: Implement answer quality evaluation
- **TODO**: Add latency and throughput monitoring
- **TODO**: Create evaluation datasets

### 🏗️ Production Readiness
- **TODO**: Add error handling and retry logic
- **TODO**: Implement caching for embeddings and results
- **TODO**: Add configuration management
- **TODO**: Support persistent Qdrant storage
- **TODO**: Add API endpoints for production deployment

### 🔒 Security and Compliance
- **TODO**: Add access control and authentication
- **TODO**: Implement audit logging
- **TODO**: Add data privacy controls
- **TODO**: Support compliance requirements (GDPR, etc.)

### 🎯 Advanced Features
- **TODO**: Multi-document reasoning
- **TODO**: Citation and source attribution
- **TODO**: Query intent classification
- **TODO**: Conversational context management
- **TODO**: Custom embedding fine-tuning

## Experiment Summary

✅ **Completed Successfully:**
- Document loading and preprocessing
- Text chunking with metadata
- Vector embedding generation
- Qdrant vector database setup with upsert support
- Similarity-based retrieval
- RAG answer generation with Ollama

🎯 **Key Insights:**
- The pipeline successfully retrieves relevant context for queries
- Chunk size and overlap parameters significantly impact retrieval quality
- SentenceTransformers provides good semantic similarity matching
- Qdrant offers excellent upsert support for CDC operations
- Ollama integration enables local LLM inference

🚀 **Next Steps:**
1. Implement CDC for live document updates using Qdrant's upsert
2. Add temporal reasoning capabilities
3. Enhance retrieval with hybrid approaches
4. Build production-ready API endpoints
5. Add comprehensive evaluation metrics

This experiment provides a solid foundation for the LiveVectorLake project's evolution into a production-ready, CDC-aware RAG system with Qdrant's superior upsert capabilities.